# IMPORT MODULES

In [2]:
import pandas as pd
import numpy as np
import random
import itertools
from config import CONSUMER_KEY, REDIRECT_URI, JSON_PATH, TD_ACCOUNT
from td.client import TDClient
import json
import pprint
import datetime
from datetime import date, time, timedelta
import requests
import config
import os


tda = TDClient(client_id=CONSUMER_KEY,redirect_uri=REDIRECT_URI, credentials_path = JSON_PATH)
tda.login()

True

# STRATEGY INPUTS AND VARIABLES

In [3]:
######################################## TESTING COMBINATION OF INPUTS FOR STRATEGY  ########################################

# BET_SIZE = [.1] #FIXED
# STOP = [.05,.1,.15] #FIXED
# TARGET = [.05,.15,.2] #FIXED

# VOL_SPIKE_THRESHOLD = [10,15,20] #Abnormally high volume that stands out on a chart
# PRICE_SPIKE_THRESHOLD = [.05,.1,.15] #Must move the price x%
# TIME_SIG_THRESHOLD = [time(hour=11,minute=0,second=0), time(hour=12,minute=0,second=0),time(hour=15,minute=30,second=0)]
# SELL_TIME_THRESHOLD = [time(hour = 11,minute = 0, second = 0), time(hour=12,minute=0,second=0),time(hour=15,minute=30,second=0)]

####################################### CREATE VARIABLES FOR INPUT STRATEGY TO TEST ##########################
ACCOUNT_SIZE = 26000
ALLOCATION = .08

bet_size_index = 0
stop_index = 1
target_index = 2
vol_spike_thresh_index = 3
price_spike_thresh_index = 4
time_sig_thresh_index = 5
sell_time_threshold = 6

#variables = [BET_SIZE,STOP,TARGET,VOL_SPIKE_THRESHOLD,PRICE_SPIKE_THRESHOLD,TIME_SIG_THRESHOLD,SELL_TIME_THRESHOLD]
#combinations = list(itertools.product(*variables))

combinations = [[.1,.05,.15,5,.05,time(hour = 12,minute = 30, second = 0),time(hour = 10,minute = 30, second = 0)]]
                

########################################  TIME FRAME  ########################################


today = datetime.datetime.now()
offset = 0

if (today.weekday() == 5) | (today.weekday() == 6):
    offset = max(1, (today.weekday() + 6) % 7 - 3) #Calculates offset to find last business day if weekend
else:
    offset = 0
    
td = datetime.timedelta(offset)

en = today - td 
st = en - timedelta(days=15)

days = (en-st).days

start_time = str(int(st.timestamp())*1000)
end_time = str(int(en.timestamp())*1000)

days = (en-st).days
days

time_interval = 30
rolling_window_days = 10
tickers_per_day = 60/time_interval*6.5
rolling_lookback = rolling_window_days*tickers_per_day

#Import the New York Stock Exchange Calendar
nyse = mcal.get_calendar('NYSE')
open_close_schedule = pd.DataFrame(nyse.schedule(start_date=st, end_date=en))
open_close_schedule.index.names = ['Date']
open_close_schedule.reset_index(inplace=True)
open_close_schedule['Date'] = open_close_schedule['Date'].dt.date
open_close_schedule['market_open'] = open_close_schedule['market_open']- timedelta(hours=4)
open_close_schedule['market_close'] = open_close_schedule['market_close']- timedelta(hours=4,minutes = time_interval)
open_close_schedule['market_open'] = open_close_schedule['market_open'].dt.time
open_close_schedule['market_close'] = open_close_schedule['market_close'].dt.time

########################################  IMPORT LIST OF ALL TICKERS FOR STRATEGY   #################################

ticker_list = pd.read_excel('/Users/beaubranton/Desktop/TRADING/DAY TRADING/Lists of Stocks/VWAP SPIKE STOCKS (High Liquidity).xlsx')
tickers = ticker_list['Ticker'].tolist()

# Run Number for Output into excel

# Get All Stock Data, Calculate Relevant Stats, and Store in DF's

In [4]:
start_clock = datetime.datetime.now() #calculate run time
go = 1
stockies = {} #Create dataframes of stock data for iteration

for ticker in tickers:
    try:
        stocks = tda.get_price_history(symbol = ticker,
                                       period_type = 'day',
                                       start_date = start_time,
                                       end_date = end_time,
                                       frequency_type='minute',
                                       frequency=time_interval,
                                       extended_hours =True)
           
        stahks = pd.DataFrame(stocks['candles'])
        
        #Skip stock if there is insufficient amount of data (data for each period of normal trading hours)
        end_check = datetime.datetime.fromtimestamp(stahks['datetime'].max()/1000)
        start_check = datetime.datetime.fromtimestamp(stahks['datetime'].min()/1000)
        daydiff = end_check.weekday() - start_check.weekday()
        days = ((end_check-start_check).days - daydiff) / 7 * 5 + min(daydiff,5) - (max(end_check.weekday() - 4, 0) % 5)
        
        print(ticker, len(stahks.index))    
        stahks['datetime'] = pd.to_datetime(stahks['datetime']/1000, unit = 's')-timedelta(hours =4)
        stahks.columns = ['Open','High','Low','Close','Volume','Datetime']
        #Rearrange Columns
        stahks['Ticker'] = ticker
        stahks['10_Day_Avg_Vol'] = stahks.Volume.rolling(int(rolling_lookback)).mean()
        stahks['Date'] = stahks['Datetime'].dt.date
        stahks['Time'] = stahks['Datetime'].dt.time
        
        #Remove any accidental duplicates (FIGURE OUT WHY????)
        stahks.drop_duplicates(['Ticker','Date','Time'],inplace = True,ignore_index=True)
            
        #Create column with the open bar's low price (For % gap up calculation with spike later in day)
        cond = (stahks['Time'] == time(hour = 9,minute = 30,second = 0))
        stahks['Day_Open_Low'] = stahks[cond].groupby('Date',as_index=True)['Low'].transform('min').ffill()
        stahks['After Hours'] = (stahks['Time'] > time(hour = 15, minute = 30, second = 0)) | (stahks['Time'] < time(hour = 9, minute = 30, second = 0))
        cond_2 = (stahks['After Hours'] == True)
        stahks['Pre-Market High'] = stahks[cond_2].groupby('Date',as_index=True)['High'].transform('max')
        
        stahks = stahks.ffill(axis=0)
        stahks = stahks.bfill(axis=0)

        
        stahks['VWAP_Row'] = stahks['Volume']*((stahks['High']+stahks['Low']+stahks['Close'])/3)
        stahks['Cum_VWAP'] = stahks.groupby('Date')['VWAP_Row'].transform('cumsum')
        stahks['Cum_Volume'] = stahks.groupby('Date')['Volume'].transform('cumsum')
        stahks['VWAP'] = stahks['Cum_VWAP']/stahks['Cum_Volume']
        stahks['VWAP_STD_1'] = stahks['VWAP'] - stahks.groupby('Date')['VWAP'].transform('std')
        stahks['Color_Bar'] = np.where(stahks['Open']<=stahks['Close'], 'Green', 'Red')
        #vol_window = 1
        stahks = stahks.merge(open_close_schedule,how = 'left', on = 'Date')
        stahks['Day_Close'] = (stahks['Time'] == stahks['market_close'])
        
        stockies[ticker] = pd.DataFrame(stahks, columns=stahks.keys())
        go += 1
        #Calculate pre-market volume for day 
        #Calculate pre-market change for the day 
        #stockies['Ticker_Return'] = (stahks['Close']/stahks['Close'].shift(vol_window))-1
        #stockies['Rolling_Vol'] = stahks['Ticker_Return'].std(ddof=130)
    except:
        pass

SFE 1
LFVN 2
ALSK 3
FNHC 4
BLNK 5
OPTN 6
ASPU 7
PICO 8
ITI 9
AIRG 10
CNTY 11
AGTC 12
EDAP 13
IDT 14
VIOT 15
JNCE 16
CECE 17
ORMP 18
SEAC 19
IIN 20
BRT 21
PRTK 22
CMCM 23
LDL 24
VTVT 25
APRN 26
MYO 27
SSSS 28
BIMI 29
MSB 30
LUMO 31
GEOS 32
RADA 33
ZAGG 34
GLAD 35
REKR 36
NVCN 37
SLGG 38
WRTC 39
HOFT 40
VRA 41
GSB 42
AQST 43
VNRX 44
HRZN 45
TCRD 46
OVID 47
SUNS 48
DFIN 49
EARN 50
LQDT 51
ACTG 52
SPRO 53
PTGX 54
CAMP 55
VNCE 56
CLCT 57
OPRX 58
IDRA 59
CSTR 60
AGFS 61
PHAS 62
JYNT 63
SACH 64
LQDA 65
AI 66
APTX 67
BRG 68
AMSC 69
ADES 70
ADMS 71
SOLY 72
ACER 73
SHSP 74
ISEE 75
GLYC 76
DAKT 77
MRAM 78
RLGT 79
AVEO 80
FARM 81
USX 82
TLRD 83
BCLI 84
LL 85
EOLS 86
VERI 87
CDTX 88
UTI 89
APYX 90
MBIO 91
SPKE 92
KERN 93
OESX 94
IMMR 95
ZYNE 96
CORR 97
CATB 98
PBPB 99
CRESY 100
MRKR 101
GPP 102
ASUR 103
ZEUS 104
DLTH 105
ASLN 106
PRTS 107
BLCM 108
CPTA 109
ATOS 110
VEL 111
PERI 112
REPH 113
NGS 114
DYAI 115
UFI 116
SLS 117
CTRN 118
TA 119
MLSS 120
FRBK 121
JAX 122
TCS 123
ALT 124
BXC 125
BPT 126
OC

# DATA PROCESSING AND SIGNALING

In [6]:
RESULT_INDEXER = 0
COMBO_INDEXER = 0
stocks_to_trade = pd.DataFrame(columns=['Strategy','Date','Ticker','Target Entry','Volume Spike','Price Spike','Previous Day Close','Signal Time', 'Shares'])

for strategy in combinations:
    print(strategy,(datetime.datetime.now() - start_clock))
    tickers = list(stockies.keys())
    for ticker in tickers:

###########################################  CALCULATE SIGNALS FOR BACKTEST  ###########################################

        #Checks for Highest Daily Volume
        high_vol_sig = np.where(stockies[ticker]['Volume'] == stockies[ticker].groupby('Date')['Volume'].transform('max'),'True','False')
        stockies[ticker]['high_vol_sig'] = high_vol_sig
        #Checks for price spike 10x greater than 
        price_sig = np.where((stockies[ticker]['High']-stockies[ticker]['Day_Open_Low'])/stockies[ticker]['Day_Open_Low'] >= strategy[price_spike_thresh_index],'True','False')
        stockies[ticker]['price_sig'] = price_sig
        #Checks that the spike was before 12 pm (tied to highest daily volume)
        before_time = np.where(stockies[ticker]['Time'] <= strategy[time_sig_thresh_index], 'True','False')
        stockies[ticker]['before_time'] = before_time 
        #Checks that spike was a green bar
        green_bar = np.where(stockies[ticker]['Color_Bar'] == 'Green', 'True','False')
        stockies[ticker]['green_bar'] = green_bar
        #Checks for volume spike 10x greater than 10day average
        vol_spike_sig = np.where(stockies[ticker]['Volume'] > strategy[vol_spike_thresh_index] * stockies[ticker]['10_Day_Avg_Vol'],'True','False')
        stockies[ticker]['vol_spike_sig'] = vol_spike_sig
        #Checks that the spike high was the highest price of the day
        high_price_sig = np.where(stockies[ticker]['High'] >= stockies[ticker].groupby('Date')['Close'].transform('max'),'True','False')
        stockies[ticker]['high_price_sig'] = high_price_sig
        #Checks to see if it time is equal to the sell_time threshold set by user
        stockies[ticker]['sell_time'] = np.where(stockies[ticker]['Time'] == strategy[sell_time_threshold],'True','False')

        #Checks all conditions
        stockies[ticker]['Grab_Price_Signal'] = np.where((high_vol_sig == 'True') & (vol_spike_sig == 'True') & (price_sig == 'True') & (high_price_sig == 'True') & (green_bar == 'True') & (before_time == 'True'),'True','False')

    ################################ INPUT BUY AND SELL SIGNALS  ##################################################

        #Find a way to turn on and of signals and rules to be 'True' 'False' 'Ignore' - yet still works with backtest framework

        #Checks that the close was lower than the VWAP (maybe in backtest)
        stockies[ticker]['Close_Condition'] = 'False'

        #Temp Variables for backtest (row by row iteration)
        #Create temporary signal day variable that signaled whether or not the price_to_buy_signal was triggered during that day
        TEMP_SIGNAL_DAY = stockies[ticker]['Date'][0] - timedelta(days=10)

        #Initially set not to trigger and gets set on price_buy_signa
        TARGET_ENTRY_PRICE = 0 
        TARGET_ENTRY_PRICE_2 = 0
        TEMP_SIGNAL_TIME = time(hour = 19, minute = 30, second = 0)
        
        VOLUME_SPIKE = 0
        PRICE_SPIKE = 0

        #Iterate over rows to see which rows meet the close condition and the all clear to buy signal (pending final signal: price cross)
        #Unique to this strategy's backtest. Could be a part of inserting variables and signals before BACKTEST SECTION
        
        for index, row in stockies[ticker].iterrows(): 
            if (row['Grab_Price_Signal'] == 'True') & (row['Date'] == en.date()):
                TEMP_SIGNAL_DAY = row['Date']
                TARGET_ENTRY_PRICE = row['VWAP']
                TEMP_SIGNAL_TIME = row['Time']
                VOLUME_SPIKE = row['Volume']/row['10_Day_Avg_Vol']
                PRICE_SPIKE = (row['High']-row['Day_Open_Low'])/row['Day_Open_Low']
                
            if (row['Close']<=TARGET_ENTRY_PRICE) & (row['Day_Close'] == True) & (row['Date'] == TEMP_SIGNAL_DAY):
                print('Got in thur')
                stocks_to_trade.at[RESULT_INDEXER,'Previous Day Close'] = row['Close']
                stocks_to_trade.at[RESULT_INDEXER,'Strategy'] = COMBO_INDEXER
                stocks_to_trade.at[RESULT_INDEXER,'Ticker'] = row['Ticker']
                stocks_to_trade.at[RESULT_INDEXER,'Shares'] = ACCOUNT_SIZE*ALLOCATION/TARGET_ENTRY_PRICE
                stocks_to_trade.at[RESULT_INDEXER,'Target Entry'] = TARGET_ENTRY_PRICE
                stocks_to_trade.at[RESULT_INDEXER,'Date'] = row['Date']
                stocks_to_trade.at[RESULT_INDEXER,'Signal Time'] = TEMP_SIGNAL_TIME
                stocks_to_trade.at[RESULT_INDEXER,'Volume Spike'] = VOLUME_SPIKE
                stocks_to_trade.at[RESULT_INDEXER,'Price Spike'] = PRICE_SPIKE
                stocks_to_trade.at[RESULT_INDEXER,'Time Threshold'] = strategy[time_sig_thresh_index]
                stocks_to_trade.at[RESULT_INDEXER,'Sell_Time'] = strategy[sell_time_threshold]

                RESULT_INDEXER +=1
                
    COMBO_INDEXER +=1
tick_list = pd.read_excel('/Users/beaubranton/Desktop/TRADING/DAY TRADING/Lists of Stocks/VWAP SPIKE STOCKS.xlsx')
stocks_to_trade = pd.merge(stocks_to_trade,tick_list[['Ticker','Market Capitalization','Sector','Shares Float']],on = 'Ticker', how = 'left')  
stocks_to_trade['Market Capitalization'] = (stocks_to_trade['Market Capitalization'].astype(float)/1000000).astype(str) + 'M'
stocks_to_trade['Shares Float'] = (stocks_to_trade['Shares Float'].astype(float)/1000000).astype(str) + 'M'
stocks_to_trade.sort_values(by = 'Market Capitalization',ascending = True,inplace = True)
stocks_to_trade.to_excel('/Users/beaubranton/Desktop/TRADING/DAY TRADING/Strategies/VWAP Resistance/Daily Signals/'+ str(date.today()+ timedelta(days=1)) + '(High Liquidity 30m).xlsx', index = True, header=True)
print(datetime.datetime.now() - start_clock)

[0.1, 0.05, 0.15, 5, 0.05, datetime.time(12, 30), datetime.time(10, 30)] 1:13:42.926523
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
Got in thur
1:14:02.185377


In [ ]:
pd.set_option('display.max_columns', 30)
stocks_to_trade

In [43]:
print(ALLOCATION)

0.08


In [35]:
### CHECK THAT TIME IS ALIGNED WITH TDAMERITRADE AND YAHOO FINANCE
# import yfinance as yf

# s = pd.to_datetime(date.today()) - timedelta(days=15)
# e = pd.to_datetime(date.today())

# tick = stockies['Ticker'][1]
# stocks = yf.download(tickers = tick, 
#                           start= s, 
#                           end= e, 
#                           auto_adjust = True,
#                           prepost = True,
#                           interval = "30m",
#                           group_by = 'ticker',
#                           progress=True)

# stahks = stocks.reset_index()

# ###################################   CALCULATE USEFUL STATS FOR BACKTEST  ###################################

# #Rearrange Columns
# stahks['Ticker'] = ticker
# stahks['Time'] = stockies['Datetime'].dt.time
# stockies[118:]

[*********************100%***********************]  1 of 1 completed


,Datetime,Open,High,Low,Close,Volume,Ticker,Time
118,2020-05-04 14:30:00-04:00,4.6400,4.6800,4.5400,4.6200,5709,DNJR,14:30:00
119,2020-05-04 15:00:00-04:00,4.7112,4.7700,4.6570,4.7200,3743,DNJR,15:00:00
120,2020-05-04 15:30:00-04:00,4.7700,4.8651,4.6500,4.6600,23015,DNJR,15:30:00
121,2020-05-04 16:00:00-04:00,4.7000,4.7000,4.7000,4.7000,2181,DNJR,16:00:00
122,2020-05-05 09:30:00-04:00,4.8900,4.9625,4.8900,4.9300,2560,DNJR,09:30:00
123,2020-05-05 10:00:00-04:00,4.9000,5.0150,4.8609,4.9100,2118,DNJR,10:00:00
124,2020-05-05 10:30:00-04:00,5.0100,5.1399,4.7000,4.8400,29022,DNJR,10:30:00
125,2020-05-05 11:00:00-04:00,4.9200,5.0650,4.7500,5.0650,7517,DNJR,11:00:00
126,2020-05-05 11:30:00-04:00,5.0000,5.0000,4.8900,4.8900,543,DNJR,11:30:00
127,2020-05-05 12:00:00-04:00,4.8941,5.0155,4.8941,4.9653,1888,DNJR,12:00:00


,Datetime,Open,High,Low,Close,Volume,Ticker,Time
0,2020-04-21 09:00:00-04:00,2.3500,2.3800,2.3500,2.3702,0,DNJR,09:30:00
1,2020-04-21 09:30:00-04:00,2.3800,2.8500,2.3100,2.7150,144392,DNJR,10:00:00
2,2020-04-21 10:00:00-04:00,2.7101,2.7795,2.6100,2.7300,37852,DNJR,10:30:00
3,2020-04-21 10:30:00-04:00,2.7002,2.8400,2.7000,2.7700,43600,DNJR,11:00:00
4,2020-04-21 11:00:00-04:00,2.7700,2.8100,2.7000,2.7000,17908,DNJR,11:30:00
...,...,...,...,...,...,...,...,...
190,2020-05-05 14:00:00-04:00,1.5200,1.5300,1.4547,1.4547,10234,DNJR,NaN
191,2020-05-05 14:30:00-04:00,1.4800,1.4800,1.4755,1.4755,16822,DNJR,NaN
192,2020-05-05 15:00:00-04:00,1.4800,1.4800,1.4500,1.4800,3192,DNJR,NaN
193,2020-05-05 15:30:00-04:00,1.4500,1.4550,1.4400,1.4402,5457,DNJR,NaN
